In [ ]:
"""
Beach wave simulation with incidence angle θ
===========================================
Model: 2D wave equation with variable speed c(x) = sqrt(g·α·x)
       implemented via a pseudodifferential operator in psipy.

Geometry:
  x ∈ [-Lx/2, Lx/2]  — cross-shore direction (-Lx/2 = offshore, Lx/2 = beach)
  y ∈ [-Ly/2, Ly/2]  — longshore direction (periodic)

Incidence angle θ:
  θ = 0   → waves perpendicular to the beach (normal case)
  θ > 0   → oblique waves, longshore component ky = k·sin(θ) ≠ 0
  Reflection preserves ky but inverts kx → asymmetry → oblique lateral wave
"""

from solver import *
import sympy as sp
import numpy as np
from IPython.display import HTML


In [ ]:
# ── 2. Grid Setup ──────────────────────────────────────────────────────────
Lx, Ly = 10.0, 10.0   
Nx, Ny = 128, 128    
Lt, Nt = 13.0, 500   # 15 seconds long
n_frames = 300

# Create meshgrid with explicit indexing format to match gradient axes
xx, yy = np.meshgrid(np.linspace(-Lx/2, Lx/2, Nx),
                     np.linspace(-Ly/2, Ly/2, Ny), indexing='ij')



In [ ]:
# ── 1. Physical parameters ────────────────────────────────────────────────────
G         = 9.81    # gravitational acceleration (m/s^2)
ALPHA     = 0.02    # beach slope: h(x) = ALPHA * (x + X0)
X0        = 9.0    # CRITICAL: Shifts domain from [-10, 10] to [0, 20] so depth >= 0 everywhere!
LAMBDA    = 4.0     # incident wavelength (m)
SPEED_FACTOR = 1.0   # 1.0 = full WKB speed

ALPHA_DEG = 5.0    # Tilted angle (degrees) so waves collide obliquely
ALPHA_R   = np.radians(ALPHA_DEG)

X1 =  (Lx - 1) / 2.0   # blade 1 starts at x = +8 (right side)
X2 = -(Lx - 1) / 2.0   # blade 2 starts at x = -8 (left side)

In [ ]:
# ── 3. SymPy symbols ────────────────────────────────────────────────────────

x, y, t       = sp.symbols('x y t',         real=True)
xi, eta      = sp.symbols('xi eta',     real=True)
g, alpha, x0  = sp.symbols('g alpha x_0',   positive=True)
u_func = Function('u') 
u = u_func(t, x, y)

# Linear bathymetry and squared phase speed
#   h(x) = alpha·(x + x0)   →   c²(x) = g·alpha·(x + x0)
# c2 = g * alpha * (x + x0)
c2 = 3
# c2 = (3 + 0.01 * x)

# Pseudodifferential symbol of the weighted Laplace operator
#   a(x, ξ) = c²(x)·(ξ₁² + ξ₂²)
symbol = c2 * (xi**2 + eta**2)

# Numerical substitution in the symbol
symbol_num = symbol.subs({g: G, alpha: ALPHA, x0: X0})

print("symbol_num=", symbol_num)

In [ ]:
# ── 4. Wave equation ────────────────────────────────────────────────────
#
#   ∂²u/∂t² = -psiOp(a(x,ξ), u)
#
# The minus sign is correct: the pseudodifferential operator associated
# with the Laplacian has a positive symbol, so -psiOp correctly gives a
# wave equation (∂²u/∂t² = c²Δu).

equation = sp.Eq(diff(u, t, t), -psiOp(symbol_num, u))



In [ ]:
# ── 5. Initial conditions ────────────────────────────────────────────────────

def _blade_straight(xx, yy, x_center, sign):
    sigma     = LAMBDA / 6.0
    gauss     = np.exp(-((xx - x_center)**2) / (2.0 * sigma**2))
    k_h       = 10.0 / sigma
    heaviside = 1.0 / (1.0 + np.exp(-k_h * sign * (x_center - xx)))
    return gauss * heaviside

def _blade_angled(xx, yy, x_center, sign):
    x_perp     =  xx * np.cos(ALPHA_R) + yy * np.sin(ALPHA_R)
    x_center_p =  x_center * np.cos(ALPHA_R)
    sigma     = LAMBDA / 6.0
    gauss     = np.exp(-((x_perp - x_center_p)**2) / (2.0 * sigma**2))
    k_h       = 10.0 / sigma
    heaviside = 1.0 / (1.0 + np.exp(-k_h * sign * (x_center_p - x_perp)))
    return gauss * heaviside

def initial_condition_b(xx, yy):
    blade1 = _blade_straight(xx, yy, X1, sign=-1)   
    blade2 = _blade_angled  (xx, yy, X2, sign=+1)   
    return blade1 + blade2

def initial_velocity_b(xx, yy):
    h_local = G * ALPHA * (xx + X0)
    h_local = np.maximum(h_local, 0.05)
    c_local = np.sqrt(h_local)

    # Determine grid orientation: if xx is constant along axis=0, then x varies along axis=1
    # This works for both 'ij' and 'xy' indexing.
    if np.allclose(xx[:, 0], xx[0, 0]):   # first column constant -> x varies horizontally (axis=1)
        axis_x = 1
        axis_y = 0
        # Spacings
        dx = xx[0, 1] - xx[0, 0]   # difference along axis=1
        dy = yy[1, 0] - yy[0, 0]   # difference along axis=0
    else:                             # 'ij' indexing: x varies along axis=0
        axis_x = 0
        axis_y = 1
        dx = xx[1, 0] - xx[0, 0]
        dy = yy[0, 1] - yy[0, 0]

    result = np.zeros_like(xx)

    b1 = _blade_straight(xx, yy, X1, sign=-1)
    db1_dx = np.gradient(b1, dx, axis=axis_x)
    result_blade1 = c_local * db1_dx * SPEED_FACTOR

    b2 = _blade_angled(xx, yy, X2, sign=+1)
    db2_dx = np.gradient(b2, dx, axis=axis_x)
    db2_dy = np.gradient(b2, dy, axis=axis_y)
    db2_dir = np.cos(ALPHA_R) * db2_dx + np.sin(ALPHA_R) * db2_dy
    result_blade2 = c_local * db2_dir * SPEED_FACTOR

    # Apply the same scaling factor to blade2's velocity
    scale2 = 2   # adjust based on max(b1)/max(b2)
    return result_blade1 - scale2 * result_blade2

In [ ]:
# ── 5b. Plot initial conditions ──────────────────────────────────────────────
# Use the same grid as the solver (already defined in cell 2)
# xx, yy are already available with indexing='ij'

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ic = initial_condition_b(xx, yy)
im = axes[0].imshow(ic, origin='lower', 
                    extent=[-Lx/2, Lx/2, -Ly/2, Ly/2],
                    cmap='RdBu_r', aspect='auto')
axes[0].set_title('Initial elevation  η(x,y,0)')
axes[0].set_xlabel('x (cross-shore, m)')
axes[0].set_ylabel('y (longshore, m)')
plt.colorbar(im, ax=axes[0], label='η (m)')

iv = initial_velocity_b(xx, yy)
im2 = axes[1].imshow(iv, origin='lower',
                     extent=[-Lx/2, Lx/2, -Ly/2, Ly/2],
                     cmap='RdBu_r', aspect='auto')
axes[1].set_title('Initial velocity  ∂η/∂t(x,y,0)')
axes[1].set_xlabel('x (cross-shore, m)')
axes[1].set_ylabel('y (longshore, m)')
plt.colorbar(im2, ax=axes[1], label='∂η/∂t (m/s)')

plt.tight_layout()
plt.show()

In [ ]:
# ── 6. Solver instantiation and configuration ────────────────────────────────

solver = PDESolver(equation)

solver.setup(
    Lx=Lx, Ly=Ly,
    Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='periodic',   # periodic in both directions
    initial_condition=initial_condition_b,
    initial_velocity=initial_velocity_b,
    n_frames=n_frames,
    plot=True,                        # displays the symbol and dispersion relation
)



In [ ]:
# ── 7. Solution ────────────────────────────────────────────────────────────

frames = solver.solve()



In [ ]:
# ── 8. Visualization ─────────────────────────────────────────────────────────
plt.rcParams['animation.embed_limit'] = 2**128
# Animation: real component, wave front detection
ani = solver.animate(
    component='real',
    overlay=None,    # marks wave fronts (gradient maxima)
    mode='surface',      # 'imshow' faster and more readable than a 3D surface
)

# Display in a Jupyter notebook:
HTML(ani.to_jshtml())